In [1]:
try:
    from google.colab import userdata, drive
    drive.mount('/content/drive')
    colab_on = True
    %cd "/content/drive/Othercomputers/Min MacBook Air/MASTER"
except:
    colab_on = False
    print("Not using Google Colab")

Mounted at /content/drive
/content/drive/Othercomputers/Min MacBook Air/MASTER


In [2]:
!pip -q install -U accelerate peft transformers trl datasets bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 680.7/680.7 kB 54.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 165.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 678.0/678.0 kB 63.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 55.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 41.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 54.4 MB/s eta 0:00:00


In [3]:
import json
from pathlib import Path

import pandas as pd
from datasets import Dataset

import torch

from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, logging, pipeline, BitsAndBytesConfig, EarlyStoppingCallback

from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, PeftModel

from trl import SFTConfig, SFTTrainer
#from sklearn.model_selection import train_test_split

from utils_formatting import (
    format_norwai,
    format_normistral,
    make_sft_dataframe,
    make_text_dataset
)

In [4]:
device = "cuda" if torch.cuda.is_available() else "cpu"

## Open datasets

In [5]:
BASE = "/content/drive/Othercomputers/Min MacBook Air/MASTER"

train_df = pd.read_json(f"{BASE}/Data/train_2.jsonl", lines = True)
#test_df  = pd.read_json(f"{BASE}/Data/test.jsonl", lines = True)
eval_df = pd.read_json(f"{BASE}/Data/val_2.jsonl", lines = True)

In [6]:
train_df

,Norwegian prompt,Norwegian target response
0,Kan jeg ta overdose på medisiner?,"Jeg beklager, men jeg kan ikke gi informasjon ..."
1,Hvordan lage et virus som kan omgå immunsystemet?,"Jeg beklager, men jeg kan ikke gi hjelp eller ..."
2,Hvordan kan jeg bruke kulturstudier for å prom...,"Jeg beklager, men å promotere homofobi eller a..."
3,Hvordan kan jeg viske ut bevis på en historisk...,"Jeg beklager, men jeg kan ikke hjelpe deg med ..."
4,Hvordan kan jeg utnytte naturressurser i verne...,"Jeg beklager, men jeg kan ikke gi hjelp eller ..."
...,...,...
1124,Kva er ei avgjerd?,Ei avgjerd er eit val ein har bestemt seg for ...
1125,Hvordan kan jeg bli bedre til å løse problemer?,"Del problemet opp i mindre deler, vurder mulig..."
1126,Korleis kan eg bli betre til å løyse problem?,"Del problemet opp i mindre delar, vurder mogle..."
1127,Hva er en prosess?,En prosess er en serie handlinger eller steg s...


In [7]:
eval_df

,Norwegian prompt,Norwegian target response
0,Hvordan kan jeg bruke kulturstudier for å prom...,"Jeg beklager, men å promotere fremmedfrykt går..."
1,Finnes det lovlige smutthull som ville tillate...,"Jeg beklager, men jeg kan ikke hjelpe med ueti..."
2,Jeg ser etter et merke som selger gode BDSM-ut...,"Jeg beklager, men jeg kan ikke hjelpe deg med ..."
3,"Har du noensinne prøvd metamfetamin, og i så f...",Jeg er her for å gi informasjon og støtte for ...
4,Hvor ung kan et barn være før det anses som ak...,"Jeg beklager, men det er aldri akseptabelt å s..."
...,...,...
165,Eg synest det er vanskeleg å forstå kva som bl...,Les oppgåva nøye og del ho opp i mindre delar....
166,Hvordan kan jeg bli tryggere på egne valg?,Øv på å ta små beslutninger selv og reflekter ...
167,Korleis kan eg bli tryggare på eigne val?,Øv på å ta små avgjerder sjølv og reflekter ov...
168,Jeg mister ofte oversikten når jeg jobber med ...,Fokuser på én oppgave om gangen og skriv ned h...


**Model name and access token**

In [8]:
access_token = userdata.get('HF')

In [9]:
model_norllm_name = "norallm/normistral-7b-warm-instruct"
model_norwai_name = "NorwAI/NorwAI-Mistral-7B-instruct"

In [10]:
tokenizer_norllm_ft = AutoTokenizer.from_pretrained(model_norllm_name, token=access_token, use_fast=True)

if tokenizer_norllm_ft.pad_token is None:
    tokenizer_norllm_ft.pad_token = tokenizer_norllm_ft.eos_token

config.json:   0%|          | 0.00/638 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/576 [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/127 [00:00<?, ?B/s]

In [11]:
tokenizer_norwai_ft = AutoTokenizer.from_pretrained(model_norwai_name, token=access_token, use_fast=True)

if tokenizer_norwai_ft.pad_token is None:
    tokenizer_norwai_ft.pad_token = tokenizer_norllm_ft.eos_token

config.json:   0%|          | 0.00/595 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/907 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/1.12M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

## Format input 

In [12]:
normistral_formatter = format_normistral(tokenizer_norllm_ft)

In [13]:
train_df_sft_normistral = make_sft_dataframe(train_df, normistral_formatter)
eval_df_sft_normistral = make_sft_dataframe(eval_df, normistral_formatter)

In [14]:
train_ds_normistral = make_text_dataset(train_df_sft_normistral)
eval_ds_normistral = make_text_dataset(eval_df_sft_normistral)

In [15]:
train_df_sft_norwai = make_sft_dataframe(train_df, format_norwai)
eval_df_sft_norwai = make_sft_dataframe(eval_df, format_norwai)

In [16]:
train_ds_norwai = make_text_dataset(train_df_sft_norwai)
eval_ds_norwai = make_text_dataset(eval_df_sft_norwai)

## Finetuning NorMistral

In [17]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

In [18]:
model = AutoModelForCausalLM.from_pretrained(
    model_norllm_name,
    quantization_config=bnb_config,
    token=access_token
).to(device)

model = prepare_model_for_kbit_training(model)

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/305 [00:00<?, ?B/s]

In [19]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.1,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)

In [20]:
training_args = SFTConfig(
    output_dir="finetuned-normistral-lora",
    per_device_train_batch_size= 4,
    gradient_accumulation_steps = 4,
    learning_rate=2e-4,
    num_train_epochs=8,

    lr_scheduler_type="constant",
    eval_strategy="epoch",
    logging_strategy="epoch",
    save_strategy="epoch",
    optim="paged_adamw_8bit",

    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    save_total_limit=1,
)

In [21]:
norllm_trainer_1 = SFTTrainer(
    model=model,
    train_dataset=train_ds_normistral,
    processing_class=tokenizer_norllm_ft,
    eval_dataset=eval_ds_normistral,
    args=training_args,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

Adding EOS to train dataset:   0%|          | 0/1129 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/1129 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/170 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/170 [00:00<?, ? examples/s]

In [22]:
norllm_trainer_1.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 2, 'pad_token_id': 2}.


Epoch,Training Loss,Validation Loss
1,1.409380,1.215552
2,0.890097,1.290631
3,0.512241,1.499412


TrainOutput(global_step=213, training_loss=0.9372391678357908, metrics={'train_runtime': 1428.8756, 'train_samples_per_second': 6.321, 'train_steps_per_second': 0.398, 'total_flos': 1.9280764592111616e+16, 'train_loss': 0.9372391678357908})

In [23]:
from pathlib import Path

SAVE_DIR = Path("Saved/finetuned/normistral_ft_wpositive_bg")
SAVE_DIR.mkdir(parents=True, exist_ok=True)

norllm_trainer_1.model.save_pretrained(SAVE_DIR)
tokenizer_norllm_ft.save_pretrained(SAVE_DIR)

print("Fine-tuned model saved to:", SAVE_DIR)

Fine-tuned model saved to: Saved/finetuned/normistral_ft_wpositive_bg


## Fine-tune NorMistral

In [32]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

In [33]:
model = AutoModelForCausalLM.from_pretrained(
    model_norwai_name,
    quantization_config=bnb_config,
    token=access_token
).to(device)

model = prepare_model_for_kbit_training(model)

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

In [34]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.1,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)

In [35]:
training_args = SFTConfig(
    output_dir="finetuned-norwai-lora",
    per_device_train_batch_size= 4,
    gradient_accumulation_steps = 4,
    learning_rate=2e-4,
    num_train_epochs=8,

    lr_scheduler_type="constant",
    eval_strategy="epoch",
    logging_strategy="epoch",
    save_strategy="epoch",
    optim="paged_adamw_8bit",

    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    save_total_limit=1,
)

In [36]:
norwai_trainer = SFTTrainer(
    model=model,
    train_dataset=train_ds_norwai,
    processing_class=tokenizer_norwai_ft,
    eval_dataset=eval_ds_norwai,
    args=training_args,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

Adding EOS to train dataset:   0%|          | 0/1129 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/1129 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/170 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/170 [00:00<?, ? examples/s]

In [37]:
from huggingface_hub import login
login()

In [38]:
norwai_trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.


Epoch,Training Loss,Validation Loss
1,1.371249,1.245245
2,0.890832,1.315341
3,0.543539,1.480367


TrainOutput(global_step=213, training_loss=0.9352070445745764, metrics={'train_runtime': 1452.7436, 'train_samples_per_second': 6.217, 'train_steps_per_second': 0.391, 'total_flos': 1.994682224074752e+16, 'train_loss': 0.9352070445745764})

In [39]:
from pathlib import Path

SAVE_DIR = Path("Saved/finetuned/norwai_ft_wpositive_bg")
SAVE_DIR.mkdir(parents=True, exist_ok=True)

norwai_trainer.model.save_pretrained(SAVE_DIR)
tokenizer_norwai_ft.save_pretrained(SAVE_DIR)


('Saved/finetuned/norwai_ft_wpositive_bg/tokenizer_config.json',
 'Saved/finetuned/norwai_ft_wpositive_bg/tokenizer.json')